# 📊 Coupling Metrics — Interactive Plotly Visualizations

**Interactive 3D plots** you can rotate, zoom, and embed in your blog!

## Features
- 🔄 **Rotate & Zoom** — Click and drag to explore
- 📱 **Responsive** — Works on mobile
- 🌐 **Embeddable** — Export as HTML for your Jekyll site
- 🎨 **Customizable** — Modify colors, camera angles, etc.

## Metrics
1. **Instability (I)** = Ce / (Ce + Ca)
2. **Abstractness (A)** = Na / Nc
3. **Distance (D)** = |A + I - 1|

In [17]:
# ============================================================
# SETUP — Run this first!
# ============================================================

# Install plotly if needed (uncomment):
# !pip install plotly pandas

import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import numpy as np
import pandas as pd

print("✅ Plotly version:", go.__version__ if hasattr(go, '__version__') else 'installed')
print("✅ Setup complete!")

✅ Plotly version: installed
✅ Setup complete!


In [18]:
# ============================================================
# 🎨 CONFIGURATION
# ============================================================

CONFIG = {
    # Background colors
    'paper_bg': '#FCF5E5',
    'plot_bg': 'rgba(252,245,229,0.8)',

    # Instability colorscale (list of [position, color])
    # Position: 0.0 to 1.0, Color: any CSS color
    'instability_colorscale': [
        [0.00, '#FFFFFF'],   # White at I=0
        [0.05, '#6AD8D8'],   # Light teal
        [0.20, '#0D3D3D'],   # Dark teal
        [0.30, '#E0B060'],   # Light bronze
        [0.45, '#5A3A12'],   # Dark bronze
        [0.50, '#3A0808'],   # Darkest red (center)
        [0.55, '#E89048'],   # Light copper
        [0.70, '#6A3210'],   # Dark copper
        [0.80, '#8A4A9A'],   # Light purple
        [0.95, '#2A0A3A'],   # Dark purple
        [1.00, '#000000'],   # Black at I=1
    ],

    # Distance colorscale (green=good, red=bad)
    'distance_colorscale': [
        [0.0, '#2E7D32'],    # Dark green (D=0, ideal)
        [0.2, '#4CAF50'],    # Green
        [0.4, '#8BC34A'],    # Light green
        [0.5, '#FFEB3B'],    # Yellow
        [0.7, '#FF9800'],    # Orange
        [0.85, '#F44336'],   # Red
        [1.0, '#B71C1C'],    # Dark red (D=1, worst)
    ],

    # Zone colors for Main Sequence
    'zone_pain': 'rgba(232,180,180,0.6)',
    'zone_useless': 'rgba(180,180,232,0.6)',
    'main_seq': '#4CAF50',

    # Default camera for 3D plots
    'camera': dict(
        eye=dict(x=1.5, y=1.5, z=1.2),
        up=dict(x=0, y=0, z=1)
    ),

    # Mesh resolution
    'mesh_points': 100,  # Lower than matplotlib for performance
}

print("🎨 Configuration loaded")

🎨 Configuration loaded


In [19]:
# ============================================================
# 🔧 HELPER FUNCTIONS
# ============================================================

def calc_instability(Ce, Ca):
    """I = Ce / (Ce + Ca)"""
    with np.errstate(divide='ignore', invalid='ignore'):
        I = Ce / (Ce + Ca)
        I = np.where((Ce == 0) & (Ca == 0), 0.5, I)
    return I

def calc_distance(A, I):
    """D = |A + I - 1|"""
    return np.abs(A + I - 1)

def create_exponential_mesh(max_val, n_points, exponent=2.5):
    """Create mesh with more points near origin"""
    t = np.linspace(0, 1, n_points)
    return (t ** exponent) * max_val

print("🔧 Helper functions loaded")

🔧 Helper functions loaded


---
## 📈 Interactive 3D: Instability Surface

**I = Ce / (Ce + Ca)**

🔄 **Drag to rotate** | 🔍 **Scroll to zoom** | 📍 **Hover for values**

In [14]:
# ============================================================
# 📈 INSTABILITY 3D SURFACE
# ============================================================

# === PARAMETERS ===
MAX_COUPLING = 20
MESH_POINTS = CONFIG['mesh_points']
MESH_EXPONENT = 2.5

# Build mesh
axis_vals = create_exponential_mesh(MAX_COUPLING, MESH_POINTS, MESH_EXPONENT)
Ca, Ce = np.meshgrid(axis_vals, axis_vals)
I = calc_instability(Ce, Ca)

# Create figure
fig_instability = go.Figure(data=[
    go.Surface(
        x=Ca, y=Ce, z=I,
        colorscale=CONFIG['instability_colorscale'],
        cmin=0, cmax=1,
        colorbar=dict(
            title=dict(text='I (Instability)', font=dict(size=14)),
            tickvals=[0, 0.25, 0.5, 0.75, 1.0],
            ticktext=['0.0 (Stable)', '0.25', '0.5', '0.75', '1.0 (Volatile)'],
            len=0.75,
        ),
        hovertemplate='Ca: %{x:.1f}<br>Ce: %{y:.1f}<br>I: %{z:.3f}<extra></extra>',
        lighting=dict(ambient=0.6, diffuse=0.8, specular=0.2, roughness=0.5),
        lightposition=dict(x=0, y=0, z=2),
    )
])

fig_instability.update_layout(
    title=dict(
        text='<b>Instability Surface: I = Ce / (Ce + Ca)</b>',
        font=dict(size=18),
        x=0.5,
    ),
    scene=dict(
        xaxis=dict(title='Ca (Afferent)', tickfont=dict(size=14, weight='bold')),
        yaxis=dict(title='Ce (Efferent)', tickfont=dict(size=14, weight='bold')),
        zaxis=dict(title='I (Instability)', tickfont=dict(size=14, weight='bold'), range=[0, 1]),
        camera=CONFIG['camera'],
        bgcolor=CONFIG['plot_bg'],
    ),
    paper_bgcolor=CONFIG['paper_bg'],
    margin=dict(l=0, r=0, t=50, b=0),
    height=700,
)

fig_instability.show()

# Uncomment to save as HTML:
# fig_instability.write_html('instability-3d-interactive.html', include_plotlyjs='cdn')

---
## 🧊 Interactive 3D: Distance from Main Sequence

**D = |A + I - 1|**

The "valley" along the diagonal is the **Main Sequence** where D = 0 (ideal).

In [15]:
# ============================================================
# 🧊 DISTANCE 3D SURFACE
# ============================================================

# Build mesh
n = CONFIG['mesh_points']
I_range = np.linspace(0, 1, n)
A_range = np.linspace(0, 1, n)
I_grid, A_grid = np.meshgrid(I_range, A_range)
D_grid = calc_distance(A_grid, I_grid)

# Main Sequence line (D=0)
main_seq_i = np.linspace(0, 1, 50)
main_seq_a = 1 - main_seq_i
main_seq_d = np.zeros_like(main_seq_i)

# Create figure
fig_distance = go.Figure()

# Surface
fig_distance.add_trace(go.Surface(
    x=I_grid, y=A_grid, z=D_grid,
    colorscale=CONFIG['distance_colorscale'],
    cmin=0, cmax=1,
    colorbar=dict(
        title=dict(text='D (Distance)', font=dict(size=14)),
        tickvals=[0, 0.25, 0.5, 0.75, 1.0],
        ticktext=['0.0 (Ideal)', '0.25', '0.5', '0.75', '1.0 (Worst)'],
        len=0.75,
    ),
    hovertemplate='I: %{x:.2f}<br>A: %{y:.2f}<br>D: %{z:.3f}<extra></extra>',
    opacity=0.9,
))

# Main Sequence line
fig_distance.add_trace(go.Scatter3d(
    x=main_seq_i, y=main_seq_a, z=main_seq_d,
    mode='lines',
    line=dict(color='white', width=8),
    name='Main Sequence (D=0)',
    hovertemplate='Main Sequence<br>I: %{x:.2f}<br>A: %{y:.2f}<extra></extra>',
))

fig_distance.update_layout(
    title=dict(
        text='<b>Distance from Main Sequence: D = |A + I - 1|</b>',
        font=dict(size=18),
        x=0.5,
    ),
    scene=dict(
        xaxis=dict(title='I (Instability)', range=[0, 1]),
        yaxis=dict(title='A (Abstractness)', range=[0, 1]),
        zaxis=dict(title='D (Distance)', range=[0, 1]),
        camera=CONFIG['camera'],
        bgcolor=CONFIG['plot_bg'],
    ),
    paper_bgcolor=CONFIG['paper_bg'],
    margin=dict(l=0, r=0, t=50, b=0),
    height=700,
    showlegend=True,
    legend=dict(x=0.02, y=0.98),
)

fig_distance.show()

# Uncomment to save:
# fig_distance.write_html('distance-3d-interactive.html', include_plotlyjs='cdn')

---
## 📊 Interactive 2D: Main Sequence Plot

**Main Sequence: A + I = 1**

Hover over points to see their Distance value.

In [16]:
# ============================================================
# 📊 MAIN SEQUENCE 2D PLOT
# ============================================================

# Example modules to plot (modify this!)
EXAMPLE_MODULES = [
    {'name': 'Database Schema', 'I': 0.1, 'A': 0.1, 'color': 'red'},
    {'name': 'Unused Interfaces', 'I': 0.9, 'A': 0.9, 'color': 'blue'},
    {'name': 'Domain Logic', 'I': 0.3, 'A': 0.7, 'color': 'green'},
    {'name': 'Service Layer', 'I': 0.7, 'A': 0.3, 'color': 'green'},
    {'name': 'Perfect Balance', 'I': 0.5, 'A': 0.5, 'color': 'darkgreen'},
    {'name': 'API Gateway', 'I': 0.8, 'A': 0.4, 'color': 'orange'},
]

# Calculate D for each
for m in EXAMPLE_MODULES:
    m['D'] = abs(m['A'] + m['I'] - 1)

fig_main_seq = go.Figure()

# Zone of Pain (lower-left triangle)
fig_main_seq.add_trace(go.Scatter(
    x=[0, 0, 0.5, 0], y=[0, 0.5, 0, 0],
    fill='toself',
    fillcolor=CONFIG['zone_pain'],
    line=dict(width=0),
    name='Zone of Pain',
    hoverinfo='name',
))

# Zone of Uselessness (upper-right triangle)
fig_main_seq.add_trace(go.Scatter(
    x=[1, 1, 0.5, 1], y=[1, 0.5, 1, 1],
    fill='toself',
    fillcolor=CONFIG['zone_useless'],
    line=dict(width=0),
    name='Zone of Uselessness',
    hoverinfo='name',
))

# Main Sequence line
fig_main_seq.add_trace(go.Scatter(
    x=[0, 1], y=[1, 0],
    mode='lines',
    line=dict(color=CONFIG['main_seq'], width=4),
    name='Main Sequence (A+I=1)',
))

# Distance contour lines
for d in [0.2, 0.4, 0.6, 0.8]:
    # Above main sequence
    x_above = [max(0, d), min(1, 1)]
    y_above = [min(1, 1-x_above[0]+d), max(0, 1-x_above[1]+d)]
    fig_main_seq.add_trace(go.Scatter(
        x=x_above, y=y_above,
        mode='lines',
        line=dict(color='gray', width=1, dash='dash'),
        showlegend=False,
        hoverinfo='skip',
    ))
    # Below main sequence
    x_below = [0, min(1, 1-d)]
    y_below = [max(0, 1-d), 0]
    fig_main_seq.add_trace(go.Scatter(
        x=x_below, y=y_below,
        mode='lines',
        line=dict(color='gray', width=1, dash='dash'),
        showlegend=False,
        hoverinfo='skip',
    ))

# Plot example modules
for m in EXAMPLE_MODULES:
    fig_main_seq.add_trace(go.Scatter(
        x=[m['I']], y=[m['A']],
        mode='markers+text',
        marker=dict(size=15, color=m['color'], line=dict(width=2, color='white')),
        text=[m['name']],
        textposition='top right',
        textfont=dict(size=11),
        name=m['name'],
        hovertemplate=(
            f"<b>{m['name']}</b><br>"
            f"I: {m['I']:.2f}<br>"
            f"A: {m['A']:.2f}<br>"
            f"D: {m['D']:.2f}<extra></extra>"
        ),
    ))

# Zone labels
fig_main_seq.add_annotation(
    x=0.15, y=0.15, text='<b>ZONE OF PAIN</b><br>(Stable + Concrete)',
    showarrow=False, font=dict(size=12, color='#8B0000'),
)
fig_main_seq.add_annotation(
    x=0.85, y=0.85, text='<b>ZONE OF USELESSNESS</b><br>(Unstable + Abstract)',
    showarrow=False, font=dict(size=12, color='#00008B'),
)

fig_main_seq.update_layout(
    title=dict(
        text='<b>Main Sequence: A + I = 1</b>',
        font=dict(size=18),
        x=0.5,
    ),
    xaxis=dict(
        title='I (Instability) →',
        range=[-0.05, 1.05],
        dtick=0.1,
        gridcolor='rgba(0,0,0,0.1)',
    ),
    yaxis=dict(
        title='A (Abstractness) →',
        range=[-0.05, 1.05],
        dtick=0.1,
        scaleanchor='x',
        gridcolor='rgba(0,0,0,0.1)',
    ),
    paper_bgcolor=CONFIG['paper_bg'],
    plot_bgcolor=CONFIG['paper_bg'],
    height=700,
    showlegend=True,
    legend=dict(x=1.02, y=1),
)

fig_main_seq.show()

# Uncomment to save:
# fig_main_seq.write_html('main-sequence-interactive.html', include_plotlyjs='cdn')

---
## 🗺️ Interactive 2D: Distance Heatmap

Hover to see exact D values at any point.

In [11]:
# ============================================================
# 🗺️ DISTANCE HEATMAP
# ============================================================

fig_heatmap = go.Figure()

# Heatmap
fig_heatmap.add_trace(go.Heatmap(
    x=I_range, y=A_range, z=D_grid,
    colorscale=CONFIG['distance_colorscale'],
    zmin=0, zmax=1,
    colorbar=dict(
        title='D',
        tickvals=[0, 0.25, 0.5, 0.75, 1.0],
    ),
    hovertemplate='I: %{x:.2f}<br>A: %{y:.2f}<br>D: %{z:.3f}<extra></extra>',
))

# Main Sequence line
fig_heatmap.add_trace(go.Scatter(
    x=[0, 1], y=[1, 0],
    mode='lines',
    line=dict(color='white', width=4),
    name='Main Sequence',
))

# Contour lines
fig_heatmap.add_trace(go.Contour(
    x=I_range, y=A_range, z=D_grid,
    contours=dict(
        start=0.1, end=0.9, size=0.2,
        showlabels=True,
        labelfont=dict(size=12, color='white'),
    ),
    line=dict(width=2, color='white'),
    showscale=False,
    hoverinfo='skip',
))

fig_heatmap.update_layout(
    title=dict(
        text='<b>Distance from Main Sequence: D = |A + I - 1|</b>',
        font=dict(size=18),
        x=0.5,
    ),
    xaxis=dict(title='I (Instability)', range=[0, 1]),
    yaxis=dict(title='A (Abstractness)', range=[0, 1], scaleanchor='x'),
    paper_bgcolor=CONFIG['paper_bg'],
    height=650,
)

fig_heatmap.show()

# Uncomment to save:
# fig_heatmap.write_html('distance-heatmap-interactive.html', include_plotlyjs='cdn')

---
## 🔄 Interactive: Instability Projections

Floor and wall projections in 3D space.

In [12]:
# ============================================================
# 🔄 INSTABILITY PROJECTIONS
# ============================================================

# Build data for projections
n = 80
ca_range = np.linspace(0, 20, n)
ce_range = np.linspace(0, 20, n)
Ca_floor, Ce_floor = np.meshgrid(ca_range, ce_range)
I_floor = calc_instability(Ce_floor, Ca_floor)

fig_proj = go.Figure()

# Floor projection (Ca × Ce at z=0)
fig_proj.add_trace(go.Surface(
    x=Ca_floor, y=Ce_floor, z=np.zeros_like(I_floor),
    surfacecolor=I_floor,
    colorscale=CONFIG['instability_colorscale'],
    cmin=0, cmax=1,
    showscale=True,
    colorbar=dict(title='I', len=0.5, y=0.75),
    opacity=0.9,
    name='Floor (Ca×Ce)',
    hovertemplate='Ca: %{x:.1f}<br>Ce: %{y:.1f}<br>I: %{surfacecolor:.3f}<extra>Floor</extra>',
))

# Back wall (Ca × I at Ce=20)
i_axis = np.linspace(0, 1, n)
Ca_back, I_back = np.meshgrid(ca_range, i_axis)
I_curve_back = 20 / (20 + ca_range)
mask_back = I_back <= np.tile(I_curve_back, (n, 1))
I_back_color = np.where(mask_back, I_back, np.nan)

fig_proj.add_trace(go.Surface(
    x=Ca_back, y=np.full_like(Ca_back, 20), z=I_back,
    surfacecolor=I_back_color,
    colorscale=CONFIG['instability_colorscale'],
    cmin=0, cmax=1,
    showscale=False,
    opacity=0.85,
    name='Back Wall (Ca×I)',
    hovertemplate='Ca: %{x:.1f}<br>I: %{z:.3f}<extra>Back Wall (Ce=20)</extra>',
))

# Left wall (Ce × I at Ca=20)
Ce_left, I_left = np.meshgrid(ce_range, i_axis)
I_curve_left = ce_range / (ce_range + 20)
mask_left = I_left <= np.tile(I_curve_left, (n, 1))
I_left_color = np.where(mask_left, I_left, np.nan)

fig_proj.add_trace(go.Surface(
    x=np.full_like(Ce_left, 20), y=Ce_left, z=I_left,
    surfacecolor=I_left_color,
    colorscale=CONFIG['instability_colorscale'],
    cmin=0, cmax=1,
    showscale=False,
    opacity=0.85,
    name='Left Wall (Ce×I)',
    hovertemplate='Ce: %{y:.1f}<br>I: %{z:.3f}<extra>Left Wall (Ca=20)</extra>',
))

# Edge curves
fig_proj.add_trace(go.Scatter3d(
    x=ca_range, y=np.full_like(ca_range, 20), z=I_curve_back,
    mode='lines', line=dict(color='black', width=4),
    name='I at Ce=20',
))
fig_proj.add_trace(go.Scatter3d(
    x=np.full_like(ce_range, 20), y=ce_range, z=I_curve_left,
    mode='lines', line=dict(color='black', width=4),
    name='I at Ca=20',
))

fig_proj.update_layout(
    title=dict(
        text='<b>Instability Projections</b><br><sub>Floor (Ca×Ce) • Back Wall (Ca×I) • Left Wall (Ce×I)</sub>',
        font=dict(size=16),
        x=0.5,
    ),
    scene=dict(
        xaxis=dict(title='Ca (Afferent)', range=[0, 20]),
        yaxis=dict(title='Ce (Efferent)', range=[0, 20]),
        zaxis=dict(title='I (Instability)', range=[0, 1]),
        camera=dict(eye=dict(x=1.8, y=1.8, z=1.0)),
        bgcolor=CONFIG['plot_bg'],
    ),
    paper_bgcolor=CONFIG['paper_bg'],
    height=700,
)

fig_proj.show()

# Uncomment to save:
# fig_proj.write_html('instability-projections-interactive.html', include_plotlyjs='cdn')

---
## 📁 Export All for Blog

Run this cell to export all visualizations as standalone HTML files.

In [ ]:
# ============================================================
# 📁 BATCH EXPORT
# ============================================================

OUTPUT_DIR = '.'  # Change to your desired output directory

exports = [
    ('instability-3d-interactive.html', fig_instability),
    ('distance-3d-interactive.html', fig_distance),
    ('main-sequence-interactive.html', fig_main_seq),
    ('distance-heatmap-interactive.html', fig_heatmap),
    ('instability-projections-interactive.html', fig_proj),
]

for filename, fig in exports:
    filepath = f"{OUTPUT_DIR}/{filename}"
    fig.write_html(filepath, include_plotlyjs='cdn')
    print(f"✅ Exported: {filepath}")

print(f"\n🎉 All {len(exports)} visualizations exported!")
print("\n📝 To embed in Jekyll:")
print('   {% include_relative instability-3d-interactive.html %}')
print("   OR use an iframe:")
print('   <iframe src="/path/to/instability-3d-interactive.html" width="100%" height="700"></iframe>')

---
## 📚 Reference

### Embedding in Jekyll

**Option 1: Include directly** (if HTML is in same directory)
```liquid
{% include_relative instability-3d-interactive.html %}
```

**Option 2: iframe** (more flexible)
```html
<iframe
  src="/assets/visualizations/instability-3d-interactive.html"
  width="100%"
  height="700"
  frameborder="0">
</iframe>
```

**Option 3: Inline with CDN** (smallest file size)
```python
fig.write_html('output.html', include_plotlyjs='cdn')  # Uses CDN
fig.write_html('output.html', include_plotlyjs=True)   # Embeds full library (~3MB)
```

### Formulas

| Metric | Formula | Meaning |
|--------|---------|--------|
| **I** | Ce / (Ce + Ca) | 0 = stable, 1 = volatile |
| **A** | abstract / total | 0 = concrete, 1 = abstract |
| **D** | \|A + I - 1\| | 0 = ideal, 1 = problematic |